# Preprocessing Pipeline
**COEN 330 — Applied Machine Learning**  
Applies and verifies all preprocessing decisions identified during EDA.  
Outputs: encoded train/val/test arrays + a saved `preprocessor.pkl`.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from preprocessing import (
    load_and_create_target, handle_outliers, apply_log_transform,
    split_data, fit_and_save_preprocessor, transform, get_feature_names
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data & Create Target

In [ ]:
df = load_and_create_target('../data/raw/dataset.csv')
print(f'Shape: {df.shape}')
df[['loan_status', 'is_risky']].value_counts()

## 2. Outlier Handling — `person_age`

Cap at 80. Values above this are invalid (EDA found a max of 144).

In [ ]:
print(f'Before — max age: {df["person_age"].max()}, rows > 80: {(df["person_age"] > 80).sum()}')
df = handle_outliers(df)
print(f'After  — max age: {df["person_age"].max()}')

## 3. Log Transform — `person_income`

In [ ]:
print(f'Before — skewness: {df["person_income"].skew():.2f}')
df = apply_log_transform(df)
print(f'After  — skewness: {df["person_income"].skew():.2f}')

fig, ax = plt.subplots(figsize=(5, 3))
sns.histplot(df['person_income'], kde=True, ax=ax, color='mediumseagreen')
ax.set_title('person_income after log1p transform')
plt.tight_layout()
plt.show()

## 4. Train / Validation / Test Split

Stratified on `is_risky` to preserve the ~78/22 class ratio in every split.  
Split: **70% train / 15% val / 15% test**.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    risky_pct = y.mean() * 100
    print(f'{name:5s} — n={len(y):6,}  |  is_risky=1: {risky_pct:.1f}%')

## 5. Encoding & Scaling

| Feature group | Columns | Strategy |
|---|---|---|
| Numerical | 8 columns | StandardScaler (fit on train only) |
| Ordinal | `person_education` | OrdinalEncoder (High School → Doctorate = 0–4) |
| Nominal | gender, home_ownership, loan_intent, prev_defaults | OneHotEncoder (drop='first') |

In [ ]:
preprocessor = fit_and_save_preprocessor(X_train, path='../models/preprocessor.pkl')

X_train_enc = transform(preprocessor, X_train)
X_val_enc   = transform(preprocessor, X_val)
X_test_enc  = transform(preprocessor, X_test)

feature_names = get_feature_names(preprocessor)

print(f'\nEncoded shape — Train: {X_train_enc.shape}')
print(f'Feature names ({len(feature_names)}): {feature_names}')

## 6. Verify — No Leakage

Confirm `loan_status` (inverse of target) is not present in any encoded array.

In [ ]:
assert 'loan_status' not in feature_names, 'DATA LEAKAGE: loan_status found in features!'
assert 'is_risky' not in feature_names, 'DATA LEAKAGE: is_risky found in features!'
print('Leakage check passed — loan_status and is_risky are not in the feature set.')

## 7. Save Processed Arrays

In [ ]:
import numpy as np

np.save('../data/processed/X_train.npy', X_train_enc)
np.save('../data/processed/X_val.npy',   X_val_enc)
np.save('../data/processed/X_test.npy',  X_test_enc)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv',     index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

import json
with open('../data/processed/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print('Processed data saved to data/processed/')

## 8. Preprocessing Summary

| Step | Decision | Justification |
|---|---|---|
| Outlier handling | Cap `person_age` at 80 | Max of 144 is biologically impossible |
| Skewness | log1p on `person_income` | Heavy right skew (median ~$67K, max ~$7.2M) |
| Drop column | `loan_status` | Inverse of target — would cause data leakage |
| Encoding | Ordinal for `person_education` | Natural ordering: High School < … < Doctorate |
| Encoding | OneHotEncoder for remaining categoricals | No natural ordering |
| Scaling | StandardScaler on numerical | Required for distance/margin-based models (SVM, k-NN) |
| Split | 70/15/15 stratified | Preserves ~78% risky class ratio; test set held out for final eval only |